In [1]:
from pathlib import Path
import json
import shutil

import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
# Пути

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

MODEL_DIRS = {
    "LogisticRegression": EXPERIMENTS_DIR / "logistic_regression",
    "DecisionTreeClassifier": EXPERIMENTS_DIR / "decision_tree",
    "LogisticRegressionTuned": EXPERIMENTS_DIR / "logistic_regression_tuned",
    "DecisionTreeClassifierTuned": EXPERIMENTS_DIR / "decision_tree_tuned",
}

BEST_MODEL_DIR = EXPERIMENTS_DIR / "best_model"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Experiments dir:", EXPERIMENTS_DIR)
print("Best model dir:", BEST_MODEL_DIR)

Experiments dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments
Best model dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\best_model


In [3]:
def load_metrics(metrics_path: Path) -> dict:
    with open(metrics_path, "r", encoding="utf-8") as file:
        return json.load(file)


metrics = []

for model_name, model_dir in MODEL_DIRS.items():
    metrics_path = model_dir / "metrics.json"
    model_metrics = load_metrics(metrics_path)
    metrics.append(model_metrics)

metrics_df = pd.DataFrame(metrics)

display(metrics_df)

,model_name,accuracy,precision,recall,f1,best_cv_f1,best_params
0,LogisticRegression,0.818966,0.777778,0.682927,0.727273,NaN,NaN
1,DecisionTreeClassifier,0.775862,0.647059,0.804878,0.717391,NaN,NaN
2,LogisticRegressionTuned,0.793103,0.673469,0.804878,0.733333,0.653360,"{'classifier__C': 1, 'classifier__class_weight..."
3,DecisionTreeClassifierTuned,0.767241,0.620690,0.878049,0.727273,0.696263,"{'class_weight': 'balanced', 'criterion': 'gin..."


In [4]:
# Сравнение моделей

comparison_columns = [
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1"
]

comparison_df = metrics_df[comparison_columns].sort_values(
    by="f1",
    ascending=False
)

display(comparison_df)

,model_name,accuracy,precision,recall,f1
2,LogisticRegressionTuned,0.793103,0.673469,0.804878,0.733333
0,LogisticRegression,0.818966,0.777778,0.682927,0.727273
3,DecisionTreeClassifierTuned,0.767241,0.620690,0.878049,0.727273
1,DecisionTreeClassifier,0.775862,0.647059,0.804878,0.717391


## Выбор лучшей модели

Лучшая модель выбирается по максимальному значению `f1-score` на validation-выборке.

Test-выборка на этапе выбора модели не используется. Она будет применена только после выбора финальной модели для честной итоговой оценки.

In [5]:
SELECTION_METRIC = "f1"

best_model_row = metrics_df.sort_values(
    by=SELECTION_METRIC,
    ascending=False
).iloc[0]

best_model_name = best_model_row["model_name"]
best_model_source_dir = MODEL_DIRS[best_model_name]

print("Best model:", best_model_name)
print("Best validation F1-score:", best_model_row[SELECTION_METRIC])
print("Best model source dir:", best_model_source_dir)

display(best_model_row.to_frame(name="value"))

Best model: LogisticRegressionTuned
Best validation F1-score: 0.7333333333333333
Best model source dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression_tuned


,value
model_name,LogisticRegressionTuned
accuracy,0.793103
precision,0.673469
recall,0.804878
f1,0.733333
best_cv_f1,0.65336
best_params,"{'classifier__C': 1, 'classifier__class_weight..."


In [6]:
# Сохраняем лучшую модель

BEST_MODEL_PATH = BEST_MODEL_DIR / "model.joblib"
BEST_VALIDATION_METRICS_PATH = BEST_MODEL_DIR / "validation_metrics.json"

shutil.copy(
    best_model_source_dir / "model.joblib",
    BEST_MODEL_PATH
)

with open(BEST_VALIDATION_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(best_model_row.to_dict(), file, indent=4)

print("Best model saved to:", BEST_MODEL_PATH)
print("Validation metrics saved to:", BEST_VALIDATION_METRICS_PATH)

Best model saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\best_model\model.joblib
Validation metrics saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\best_model\validation_metrics.json


# Проверка на test

In [7]:
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"

test_data = pd.read_csv(TEST_PATH)

TARGET_COLUMN = "Outcome"

X_test = test_data.drop(columns=[TARGET_COLUMN])
y_test = test_data[TARGET_COLUMN]

best_model = joblib.load(BEST_MODEL_PATH)

y_test_pred = best_model.predict(X_test)

test_metrics = {
    "model_name": best_model_name,
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "precision": float(precision_score(y_test, y_test_pred)),
    "recall": float(recall_score(y_test, y_test_pred)),
    "f1": float(f1_score(y_test, y_test_pred))
}

test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)

,model_name,accuracy,precision,recall,f1
0,LogisticRegressionTuned,0.801724,0.680851,0.8,0.735632


In [8]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.88      0.80      0.84        76
           1       0.68      0.80      0.74        40

    accuracy                           0.80       116
   macro avg       0.78      0.80      0.79       116
weighted avg       0.81      0.80      0.80       116



In [9]:
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

test_conf_matrix_df = pd.DataFrame(
    test_conf_matrix,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(test_conf_matrix_df)

,Predicted 0,Predicted 1
Actual 0,61,15
Actual 1,8,32


In [10]:
BEST_TEST_METRICS_PATH = BEST_MODEL_DIR / "test_metrics.json"

with open(BEST_TEST_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=4)

print("Test metrics saved to:", BEST_TEST_METRICS_PATH)

Test metrics saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\best_model\test_metrics.json


## Вывод

В результате сравнения baseline-моделей и моделей после подбора гиперпараметров лучшей моделью по `f1-score` на validation-выборке была выбрана модель `LogisticRegressionTuned`.

Выбранная модель была сохранена в директорию `experiments/best_model/` и дополнительно проверена на test-выборке.